### Train oracle model

In [2]:

from models import mlp_oracle
import torch
from torch.utils.data import DataLoader
import numpy as np
import os
import mavenn

from torch.utils.data import random_split

2026-02-25 13:49:39.136596: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
dataset = mavenn.load_example_dataset('gb1')
x = dataset['x']
y = dataset['y']


In [4]:
# One hot encode the sequences
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
vocab_size = len(AMINO_ACIDS)  # 20 amino acids
aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def sequences_to_indices(sequences):
    """Convert string sequences to 2D array of indices."""
    seq_len = len(sequences[0])
    indices = np.zeros((len(sequences), seq_len), dtype=np.int32)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            indices[i, j] = aa_to_idx.get(aa, 0)
    return indices

def one_hot_encode(sequences, vocab_size):
    """One-hot encode sequences (2D array of indices)."""
    one_hot = np.zeros((sequences.shape[0], sequences.shape[1], vocab_size), dtype=np.float32)
    for i in range(sequences.shape[0]):
        for j in range(sequences.shape[1]):
            aa_index = sequences[i, j]
            if aa_index < vocab_size:
                one_hot[i, j, aa_index] = 1.0
    return one_hot

# Convert string sequences to indices, then one-hot encode
x_indices = sequences_to_indices(x)
x_one_hot = one_hot_encode(x_indices, vocab_size)
print(f"Shape: {x_one_hot.shape}")  # Should be (n_samples, seq_len, 20)

Shape: (530737, 55, 20)


In [5]:
# Flatten one-hot encoded data: (n_samples, seq_len, vocab_size) -> (n_samples, seq_len * vocab_size)
x_flat = x_one_hot.reshape(x_one_hot.shape[0], -1)  # Shape: (n_samples, 80)
y_float = y.astype(np.float32)  # Convert to float32 for PyTorch
print(f"Flattened shape: {x_flat.shape}")


full_dataset = list(zip(x_flat, y_float))
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
print(f"Train size: {train_size}, Val size: {val_size}")

Flattened shape: (530737, 1100)
Train size: 424589, Val size: 106148


In [6]:
print(f"x_flat.shape: {x_flat.shape}")
print(f"y range: [{y.min():.2f}, {y.max():.2f}]")

x_flat.shape: (530737, 1100)
y range: [-11.20, 3.60]


In [7]:
def train_oracle_model(train_loader, val_loader, model_save_path):
    """Train the MLP oracle model on the GB1 dataset with validation and save the trained model."""

    # Initialize model - reduced hidden size for regularization
    input_size = x_flat.shape[1]
    output_size = 1
    print(f"Input size: {input_size}")
    model = mlp_oracle.MLPOracle(input_size, output_size, hidden_size=128, dropout_rate=0.5)
    
    # Fit normalization parameters (on training data only)
    print("Fitting normalization parameters...")
    model.fit_normalization(train_loader)
    
    # Train the model with validation and early stopping
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train_model(train_loader, val_loader=val_loader, device=device, 
                      epochs=200, weight_decay=1e-3, patience=20)
    
    # Save the trained model parameters
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    model.save_model(model_save_path)

train_oracle_model(train_loader, val_loader, model_save_path="models/oracle_mlp.pth")

Input size: 1100
Fitting normalization parameters...
Epoch 1/200, Train Loss: 1.5823, Val Loss: 0.8057
Epoch 2/200, Train Loss: 1.0377, Val Loss: 0.7128
Epoch 3/200, Train Loss: 0.9852, Val Loss: 0.7070
Epoch 4/200, Train Loss: 0.9629, Val Loss: 0.6910
Epoch 5/200, Train Loss: 0.9418, Val Loss: 0.6592
Epoch 6/200, Train Loss: 0.9399, Val Loss: 0.6579
Epoch 7/200, Train Loss: 0.9368, Val Loss: 0.6328
Epoch 8/200, Train Loss: 0.9378, Val Loss: 0.6055
Epoch 9/200, Train Loss: 0.9265, Val Loss: 0.6297
Epoch 10/200, Train Loss: 0.9119, Val Loss: 0.6423
Epoch 11/200, Train Loss: 0.9029, Val Loss: 0.6295
Epoch 12/200, Train Loss: 0.8995, Val Loss: 0.6719
Epoch 13/200, Train Loss: 0.8993, Val Loss: 0.6728
Epoch 14/200, Train Loss: 0.8943, Val Loss: 0.7677
Epoch 15/200, Train Loss: 0.8880, Val Loss: 0.6029
Epoch 16/200, Train Loss: 0.8805, Val Loss: 0.6080
Epoch 17/200, Train Loss: 0.8792, Val Loss: 0.6754
Epoch 18/200, Train Loss: 0.8740, Val Loss: 0.5954
Epoch 19/200, Train Loss: 0.8710, Val 

In [9]:
# test inference
print(x_flat.shape[1])
oracle = mlp_oracle.MLPOracle(input_size=x_flat.shape[1], output_size=1, hidden_size=128, dropout_rate=0.3)
oracle.load_model('models/oracle_mlp.pth')
sequence = torch.tensor(x_flat[0], dtype=torch.float32).unsqueeze(0)  # Shape: (1, 80)
prediction = oracle.inference(sequence)
print(f"Predicted binding score: {prediction.item():.4f}, True binding score: {y[0]:.4f}")

1100
Predicted binding score: -2.1408, True binding score: -3.1452


In [ ]:
# In train_Oracle.ipynb after loading
prediction = oracle.inference(torch.tensor(x_flat[:10], dtype=torch.float32))
print(f"Predictions: {prediction.squeeze().tolist()}")
print(f"True values: {y[:10].tolist()}")

In [11]:
# test inference on a mutated sequence
sequence = list(x[0])  # Original sequence as a list of characters
sequence[0] = 'V'
sequence[1] = 'V'
sequence[2] = 'V'
sequence[3] = 'V'  # Mutate the first amino acid to 'V
mutated_sequence = ''.join(sequence)  # Convert back to string
print(f"Original sequence: {x[0]}, Mutated sequence: {mutated_sequence}")
# Convert mutated sequence to one-hot encoding
mutated_indices = sequences_to_indices([mutated_sequence])  # Shape: (1, seq_len)
mutated_one_hot = one_hot_encode(mutated_indices, vocab_size)  #
mutated_flat = mutated_one_hot.reshape(mutated_one_hot.shape[0], -1)  # Shape: (1, 80)
mutated_tensor = torch.tensor(mutated_flat, dtype=torch.float32)  # Shape:
mutated_prediction = oracle.inference(mutated_tensor)
print(f"Predicted binding score for mutated sequence: {mutated_prediction.item():.4f}")

Original sequence: AAKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE, Mutated sequence: VVVVILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE
Predicted binding score for mutated sequence: 0.6857
